In [2]:
import sys

from customer_cleaner import CustomerCleaner
import pandas as pd
from sqlalchemy.engine import URL
from sqlalchemy import create_engine
import pandas as pd

user = "root"
password = "root"
host = "localhost"
port = 3306
database = "proj"

url = URL.create(
    drivername="mysql+pymysql",
    username=user,
    password=password,
    host=host,
    port=port,
    database=database
)

engine = create_engine(url=url)

def clean_staging(database_name, engine):
    df = pd.read_sql(f"SELECT * FROM {database_name}", con = engine)
    cleaner = CustomerCleaner(df)
    cleaned_df = cleaner.clean()

    return cleaned_df

def build_features(engine):
    try:
        df = pd.read_sql(
            "SELECT * FROM cleaned_customers",
            con=engine
        )

        # Feature 1 - tenure_bucket

        bins = [0, 12, 24, 48, 72]
        labels = ["0-12", "13-24", "25-48", "49-72"]
        df["tenure_bucket"] = pd.cut(
            df["tenure"],
            bins=bins,
            labels=labels,
            include_lowest=True
        )

        # Feature 2 - high_change_flag

        median = df["monthly_charges"].mean()
        def fill_high_charge(charge):
            return 1 if charge > median else 0

        df["high_charge_flag"] = df["monthly_charges"].apply(fill_high_charge)

        # Feature 3 - serivce_count

        def fill_service_count(row):
            service_columns = [
                "online_security",
                "online_backup",
                "device_protection",
                "tech_support",
                "streaming_tv",
                "streaming_movies"
            ]
            count = 0
            for cols in service_columns:
                count += 1 if row[cols] == "Yes" else 0
            return count

        df["service_count"] = df.apply(fill_service_count, axis=1)

        # Feature 4 - is_long_term_customer

        def fill_is_long_term_customer(tenure):
            return 1 if tenure >= 24 else 0

        df["is_long_term_customer"] = df["tenure"].apply(fill_is_long_term_customer)

        # Feature 5 - has_streaming_bundle

        def fill_has_streaming_bundle(row):
            return 1 if row["streaming_tv"] == "Yes" and row["streaming_movies"] == "Yes" else 0

        df["has_streaming_bundle"] = df[["streaming_tv", "streaming_movies"]].apply(fill_has_streaming_bundle, axis = 1)

        # Feature 6 - auto_pay_flag

        def fill_auto_pay_flag(value):
            return 1 if "automatic" in value else 0

        df["auto_pay_flag"] = df["payment_method"].apply(fill_auto_pay_flag)

        df = df[['customer_id', 'auto_pay_flag', 'has_streaming_bundle' , 'is_long_term_customer', 'service_count', 'high_charge_flag', 'tenure_bucket', 'churn']]

        return df
    
    except Exception as e:
        print("Error Occured:", e)

def validation(customer_ml_features, cleaned_df):

    if customer_ml_features.shape[0] == cleaned_df.shape[0]:
        print("PASS: Customer ML Features and Cleaned df row counts match")
    else:
        print("FAIL: Customer ML Features and Cleaned df row counts do not match")

    feature_columns = [
        "auto_pay_flag",
        "has_streaming_bundle",
        "is_long_term_customer",
        "service_count",
        "high_charge_flag",
        "tenure_bucket"
    ]

    missing_columns = []

    for col in feature_columns:
        if col not in customer_ml_features.columns:
            missing_columns.append(col)

    if len(missing_columns) == 0:
        print("PASS: All 6 feature columns are present")
    else:
        print("FAIL: Missing columns:", missing_columns)

    for col in feature_columns:
        null_count = customer_ml_features[col].isnull().sum()

        if null_count == 0:
            print(f"PASS: {col} has no NULL values")
        else:
            print(f"FAIL: {col} has {null_count} NULL values")

if __name__ == "__main__":

    cleaned_df = clean_staging(
        "stg_customer_raw",
        engine=engine
    )

    customer_ml_features = build_features(
        engine=engine
    )

    validation(
        customer_ml_features,
        cleaned_df
    )

    customer_ml_features.to_sql(
        "customer_ml_features_DE",
        con = engine,
        if_exists = "replace",
        index = False
    )

2026-08-17 15:48:51,220 - INFO - CustomerCleaner initialized with DataFrame shape (7043, 21)
2026-08-17 15:48:51,221 - INFO - Starting data cleaning pipeline
2026-08-17 15:48:51,221 - INFO - Standardizing column names
2026-08-17 15:48:51,225 - INFO - Column standardization completed
2026-08-17 15:48:51,225 - INFO - Fixing total_charges column
2026-08-17 15:48:51,266 - INFO - Converted total_charges to numeric. Blank rows affected: 11
2026-08-17 15:48:51,268 - INFO - Normalizing binary columns
2026-08-17 15:48:51,275 - INFO - Normalized column: partner
2026-08-17 15:48:51,282 - INFO - Normalized column: dependents
2026-08-17 15:48:51,288 - INFO - Normalized column: phone_service
2026-08-17 15:48:51,292 - INFO - Normalized column: paperless_billing
2026-08-17 15:48:51,298 - INFO - Normalized column: churn
2026-08-17 15:48:51,300 - INFO - Binary column normalization completed
2026-08-17 15:48:51,301 - INFO - Handling null values
2026-08-17 15:48:51,304 - INFO - Filled 11 null values in to

PASS: Customer ML Features and Cleaned df row counts match
PASS: All 6 feature columns are present
PASS: auto_pay_flag has no NULL values
PASS: has_streaming_bundle has no NULL values
PASS: is_long_term_customer has no NULL values
PASS: service_count has no NULL values
PASS: high_charge_flag has no NULL values
PASS: tenure_bucket has no NULL values


C:\Users\aravind.harish\AppData\Local\Temp\ipykernel_31340\2689778207.py:156: UserWarning: The provided table name 'customer_ml_features_DE' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  customer_ml_features.to_sql(
